<a href="https://colab.research.google.com/github/Aswini200303/Footing-Reinforcement-Weight-Calculator/blob/main/TestRebar_Weight__(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Footing Reinforcement Weight Calculator
**Automated Rebar Weight from Plan & Schedule PDFs**

## README – Run Instructions, Assumptions, Limitations

This notebook is designed to calculate the total rebar weight from provided 'Plan PDF' and 'Footing Schedule PDF' documents. It automates the extraction of footing marks and counts from the plan, and rebar details (quantity, bar size, length) from the schedule, processes this data, and provides a total rebar weight.

### Instructions
1.  **Run all cells sequentially**: Start from the top and execute each code cell in order.
2.  **Upload PDFs**: When prompted, upload your 'Plan PDF' and 'Footing Schedule PDF' files using the provided interactive widgets.
3.  **Review Outputs**: Observe the printed dataframes and final rebar weight. The notebook will generate `footing_counts.csv` and `rebar_weight_results.csv` files, which can be downloaded.
4.  **Gemini API Key**: Ensure your Gemini API Key is configured in Colab Secrets as `GEMINI_API_KEY` for primary PDF extraction functionality. If not configured, the notebook will fall back to `pdfplumber` where possible, but Gemini API is the recommended and primary method.

### Assumptions
*   **PDF Structure**: It is assumed that the 'Plan PDF' contains footing marks in a recognizable format (e.g., 'F1', 'F01', 'F104') that can be extracted using regular expressions or interpreted by the Gemini API.
*   **Schedule PDF Structure**: The 'Footing Schedule PDF' is assumed to contain tabular data with columns for 'Mark', 'Footing Length', 'Footing Width', 'Footing Thickness', 'Rebar Bottom', and 'Rebar Top'. The rebar information within the 'Rebar Bottom' and 'Rebar Top' columns is expected to follow a pattern like `(Qty) #BarSize` (e.g., `(6) #5`).
*   **Rebar Orientation**: Rebar specified in the schedule (e.g., 'Rebar Bottom', 'Rebar Top') is assumed to run "EACH WAY, UNO" (each way, unless noted otherwise), meaning bars are present along both the length and width of the footing, with the same quantity and bar size. This results in two entries for each rebar specification per footing (one for length-direction bars, one for width-direction bars).
*   **Units**: Footing dimensions (length, width, thickness) and rebar lengths in the schedule are assumed to be in feet-inch format (e.g., `5'-0"`, `5'-6"`, `60"`). These are converted to decimal feet for calculations.
*   **Unit Weights**: The notebook uses a comprehensive set of standard unit weights for rebar sizes #3 through #18 (lbs/ft) as defined internally.
*   **Missing Entries**: If a footing mark from the 'Plan PDF' does not have a corresponding rebar entry in the 'Footing Schedule PDF', its rebar weight contribution is explicitly assumed to be zero, and a warning is issued.
*   **Default Multiplier**: A default multiplier of 4 is used (representing top & bottom, each way) for rebar quantity calculation, unless explicitly parsed otherwise.

### Limitations
*   **PDF Parsing Robustness**: While the Gemini API is used for enhanced interpretation, and `pdfplumber` provides a fallback, PDF table extraction and text parsing can still be sensitive to highly unusual or complex PDF layouts and formatting. Significant deviations from expected structures might require manual intervention or further refinement of extraction logic.
*   **Gemini API Dependency**: The primary extraction relies on the Gemini API. Issues with API access, rate limits, or unexpected model responses can affect performance and accuracy, triggering the `pdfplumber` fallback.
*   **Complex Rebar Designs**: The notebook currently assumes rebar runs "EACH WAY, UNO" and does not account for more complex rebar configurations like stirrups, hooks, or varying bar quantities/sizes along different dimensions that might be specified in more detailed schedules beyond a simple Qty #BarSize format.
*   **Footing Thickness/Depth**: The current rebar weight calculation does not explicitly use footing thickness or depth. This assumes rebar is appropriately placed within the footing as per design, but the model does not verify this structural aspect.
*   **Image-based PDFs**: If the PDFs are image-based (scanned documents without searchable text), `pdfplumber` will fail to extract text, and Gemini's ability to interpret will be limited to its OCR capabilities.

## 1. Setup – Libraries & Gemini Key

In [ ]:
# @title Install Required Libraries
# Install PyPDF2, pdfplumber, pandas, openpyxl for PDF parsing and data handling.
# Install google-generativeai for Gemini API interaction.
!pip install PyPDF2 pdfplumber pandas openpyxl google-generativeai --quiet

print("Required libraries installed successfully.")

In [ ]:
import re
import pandas as pd
import pdfplumber
import io
from collections import Counter
import warnings
import ipywidgets as widgets
from IPython.display import display, HTML

# For Gemini API
import google.generativeai as genai
from google.colab import userdata # Only needed if API key is in Colab Secrets

print("All necessary modules imported.")

In [ ]:
print("Listing available Gemini models...")
try:
    for m in genai.list_models():
        if "generateContent" in m.supported_generation_methods:
            print(m.name)
except Exception as e:
    print(f"Error listing models: {e}")
print("Please review the list above and let me know which model you would like to use for content generation.")

In [ ]:
# @title Configure Gemini API (gemini-2.5-flash)
# User-provided API key from earlier conversation.
# For a production notebook, it is highly recommended to store API keys securely in Colab Secrets.
# To store in Colab Secrets:
# 1. Click on the '🔒' icon (Secrets) in the left panel.
# 2. Click '+ New secret'.
# 3. For 'Name', enter 'GEMINI_API_KEY'.
# 4. For 'Value', paste your Gemini API key.
# 5. Make sure 'Notebook access' is turned ON for this notebook.

# Securely fetch API key from Colab Secrets
api_key = userdata.get('GEMINI_API_KEY')
genai.configure(api_key=api_key)

# Initialize the Gemini model
# Switching to 'gemini-2.5-flash' as requested by the user.
gemini_model = genai.GenerativeModel('gemini-2.5-flash')

print("Gemini API configured and 'gemini-2.5-flash' model initialized.")

## 2. Upload PDFs (ipywidgets + validation)

In [ ]:
# @title PDF Upload Interface
# ────────────────────────────────────────────────
#        PDF Upload UI - Footing Weight Calculator
# ────────────────────────────────────────────────

# Global variables to store uploaded PDF contents and filenames
plan_pdf_bytes = None
schedule_pdf_bytes = None
plan_filename = None
schedule_filename = None

# Output widgets for displaying feedback
plan_output = widgets.Output()
schedule_output = widgets.Output()
validation_output = widgets.Output()

# 1. File Upload Widgets
plan_upload = widgets.FileUpload(
    accept='.pdf',  # Only allow PDF files
    multiple=False,  # Allow only one file upload
    description='Upload Plan PDF'
)
schedule_upload = widgets.FileUpload(
    accept='.pdf',  # Only allow PDF files
    multiple=False,  # Allow only one file upload
    description='Upload Footing Schedule PDF'
)

# 2. Function to handle Plan PDF upload feedback and storage
def on_plan_upload_change(change):
    global plan_pdf_bytes, plan_filename
    with plan_output:
        plan_output.clear_output() # Clear previous output
        if plan_upload.value: # Check if a file was uploaded
            file_info = list(plan_upload.value.values())[0]
            plan_filename = file_info['metadata']['name']
            plan_pdf_bytes = file_info['content']

            if not plan_filename.lower().endswith('.pdf'):
                display(HTML(f"<p style='color:red;'>Error: '{plan_filename}' is not a PDF file. Please upload a valid PDF.</p>"))
                plan_pdf_bytes = None # Clear content if not a PDF
            else:
                display(HTML(f"<p style='color:green;'>'Plan PDF' ('{plan_filename}') uploaded successfully. Size: {len(plan_pdf_bytes)} bytes</p>"))
        else:
            plan_pdf_bytes = None
            plan_filename = None
            display(HTML("<p style='color:orange;'>No Plan PDF uploaded yet.</p>"))

# 3. Function to handle Footing Schedule PDF upload feedback and storage
def on_schedule_upload_change(change):
    global schedule_pdf_bytes, schedule_filename
    with schedule_output:
        schedule_output.clear_output() # Clear previous output
        if schedule_upload.value: # Check if a file was uploaded
            file_info = list(schedule_upload.value.values())[0]
            schedule_filename = file_info['metadata']['name']
            schedule_pdf_bytes = file_info['content']

            if not schedule_filename.lower().endswith('.pdf'):
                display(HTML(f"<p style='color:red;'>Error: '{schedule_filename}' is not a PDF file. Please upload a valid PDF.</p>"))
                schedule_pdf_bytes = None # Clear content if not a PDF
            else:
                display(HTML(f"<p style='color:green;'>'Footing Schedule PDF' ('{schedule_filename}') uploaded successfully. Size: {len(schedule_pdf_bytes)} bytes</p>"))
        else:
            schedule_pdf_bytes = None
            schedule_filename = None
            display(HTML("<p style='color:orange;'>No Footing Schedule PDF uploaded yet.</p>"))

# Attach change handlers to the upload widgets
plan_upload.observe(on_plan_upload_change, names='value')
schedule_upload.observe(on_schedule_upload_change, names='value')

# 4. Validate & Proceed Button
validate_button = widgets.Button(
    description='Validate & Proceed',
    button_style='success', # Green button
    icon='check' # A check icon
)

# 5. Function to handle validation logic
def on_validate_button_click(b):
    with validation_output:
        validation_output.clear_output() # Clear previous validation messages
        all_ok = True

        if plan_pdf_bytes is None:
            display(HTML("<p style='color:red;'>Error: Plan PDF is not uploaded yet.</p>"))
            all_ok = False
        elif not plan_filename.lower().endswith('.pdf'):
            display(HTML(f"<p style='color:red;'>Error: Plan PDF ('{plan_filename}') is not a valid PDF.</p>"))
            all_ok = False

        if schedule_pdf_bytes is None:
            display(HTML("<p style='color:red;'>Error: Footing Schedule PDF is not uploaded yet.</p>"))
            all_ok = False
        elif not schedule_filename.lower().endswith('.pdf'):
            display(HTML(f"<p style='color:red;'>Error: Footing Schedule PDF ('{schedule_filename}') is not a valid PDF.</p>"))
            all_ok = False

        if all_ok:
            display(HTML("<div style='background-color:#e6ffe6; border: 1px solid #00cc00; padding: 10px; border-radius: 5px; margin-top: 10px;'>" \
                           "<p style='color:#008000; font-weight:bold;'>Both files uploaded successfully. You can now run the next cells to extract counts and calculate rebar weight.</p></div>"))
        else:
            display(HTML("<p style='color:red; font-weight:bold; margin-top: 10px;'>Please resolve the upload errors before proceeding.</p>"))

# Attach click event to the validate button
validate_button.on_click(on_validate_button_click)

# 6. Layout and Display UI
title_html = widgets.HTML("<h2><i class='fa fa-upload'></i> Footing Reinforcement Data Upload</h2>")
description_html = widgets.HTML("<p>Please upload the 'Plan PDF' and 'Footing Schedule PDF' files below. The system will validate the uploads and prepare them for processing.</p>")

plan_upload_box = widgets.VBox([
    widgets.Label("1. Plan PDF:", layout=widgets.Layout(font_weight='bold')),
    plan_upload,
    plan_output
], layout=widgets.Layout(border='1px solid lightgray', padding='10px', margin='5px', width='48%'))

schedule_upload_box = widgets.VBox([
    widgets.Label("2. Footing Schedule PDF:", layout=widgets.Layout(font_weight='bold')),
    schedule_upload,
    schedule_output
], layout=widgets.Layout(border='1px solid lightgray', padding='10px', margin='5px', width='48%'))

upload_section = widgets.HBox([plan_upload_box, schedule_upload_box])

validation_section = widgets.VBox([
    validate_button,
    validation_output
], layout=widgets.Layout(border='1px solid lightgray', padding='10px', margin='5px', width='98%'))

full_ui = widgets.VBox([title_html, description_html, upload_section, validation_section])

display(full_ui)

## 3. Q1 – Footing Counts (Gemini first + pdfplumber fallback)

In [ ]:
import json

# Ensure plan_pdf_bytes is available from the upload UI
if plan_pdf_bytes is None:
    raise ValueError("Plan PDF content not found. Please upload the Plan PDF using the UI above and click 'Validate & Proceed'.")

print("\nAttempting to extract footing counts using Gemini API...")

# --- PDFPlumber initial extraction (always run for completeness) ---
all_footing_marks = []
try:
    with pdfplumber.open(io.BytesIO(plan_pdf_bytes)) as pdf:
        for page in pdf.pages:
            page_text = page.extract_text()
            if page_text:
                found_marks = re.findall(r'F\d+', page_text) # Improved regex: 'F' followed by one or more digits
                all_footing_marks.extend(found_marks)
    pdfplumber_footing_counts = Counter(all_footing_marks)
    print("\033[92mpdfplumber initial extraction successful for footing marks.\033[0m")
except Exception as e:
    print(f"\033[91mError during pdfplumber initial extraction for footing counts: {e}. Footing counts may be incomplete.\033[0m")
    pdfplumber_footing_counts = {}

# Start with pdfplumber's counts as the baseline
footing_counts = pdfplumber_footing_counts.copy()

# --- Attempt Gemini API for potential refinement/cross-check ---
try:
    prompt_parts = [
        "You are an expert civil engineer. Extract ALL footing marks (e.g., F1, F01, F104) and their corresponding counts from the provided Plan PDF document. Count the occurrences of each unique footing mark.",
        "Return the data as a strict JSON object with a single top-level key 'counts' whose value is an object mapping each footing mark to its count. Do NOT include any markdown formatting (like ```json) or extra text. Only the JSON object.",
        {"mime_type": "application/pdf", "data": plan_pdf_bytes}
    ]

    response = gemini_model.generate_content(prompt_parts)

    if response.text and response.text.strip():
        try:
            gemini_output = json.loads(response.text)
            if 'counts' in gemini_output and isinstance(gemini_output['counts'], dict):
                gemini_footing_counts = {k: int(v) for k, v in gemini_output['counts'].items()}
                print("\033[92mGemini API extraction successful for footing counts.\033[0m")

                # Merge Gemini's counts, prioritizing Gemini if there's a conflict or more complete
                # For now, let's update with Gemini's, assuming it might be more accurate if successful.
                # A more sophisticated merge could compare counts for existing keys.
                footing_counts.update(gemini_footing_counts)
            else:
                print("\033[93mWarning: Gemini API output was not in the expected 'counts' JSON format. Using pdfplumber's counts.\033[0m")
        except json.JSONDecodeError:
            print("\033[93mWarning: Gemini API returned invalid JSON. Using pdfplumber's counts.\033[0m")
            print(f"Gemini raw response: {response.text[:200]}...") # Print first 200 chars for debugging
    else:
        print("\033[93mWarning: Gemini API returned empty response. Using pdfplumber's counts.\033[0m")

except Exception as e:
    print(f"\033[91mError during Gemini API call for footing counts: {e}. Using pdfplumber's counts.\033[0m")

# Create a pandas DataFrame from the consolidated counts
if footing_counts:
    footing_counts_df = pd.DataFrame(footing_counts.items(), columns=['Mark', 'Count'])
    footing_counts_df = footing_counts_df.sort_values(by='Mark').reset_index(drop=True)

    footing_counts_df.to_csv('footing_counts.csv', index=False)
    print("Footing counts saved to 'footing_counts.csv'.")
    print("\nAll Footing Counts:")
    display(footing_counts_df)
else:
    print("\033[91mNo footing marks could be extracted from the Plan PDF.\033[0m")
    footing_counts_df = pd.DataFrame(columns=['Mark', 'Count'])

### Inspecting Raw Text from Plan PDF (using pdfplumber)
Let's extract and display all text that `pdfplumber` can read from your 'Plan PDF' to verify if the missing footing marks (F01, F101, F102) are present in the raw extracted text.

In [ ]:
import pdfplumber
import io

if plan_pdf_bytes is None:
    print("Plan PDF content not found. Please upload the Plan PDF first.")
else:
    extracted_text = []
    try:
        with pdfplumber.open(io.BytesIO(plan_pdf_bytes)) as pdf:
            for i, page in enumerate(pdf.pages):
                page_text = page.extract_text()
                if page_text:
                    extracted_text.append(f"--- Page {i+1} ---\n{page_text}")
                else:
                    extracted_text.append(f"--- Page {i+1} (No text extracted) ---")

        full_text = "\n".join(extracted_text)
        print("\nFull text extracted from Plan PDF by pdfplumber:\n")
        print(full_text)

        # Check for specific footing marks the user mentioned
        print("\n--- Verification for User-mentioned Footings ---")
        missing_marks_to_check = ['F01', 'F101', 'F102']
        for mark in missing_marks_to_check:
            if mark in full_text:
                print(f"'{mark}' *found* in extracted text.")
            else:
                print(f"'{mark}' *not found* in extracted text.")

    except Exception as e:
        print(f"Error extracting text with pdfplumber: {e}")

## 4. Q2 – Schedule Rebar Details (Gemini first + pdfplumber fallback)

In [ ]:
# @title Extract and Parse Rebar Details from Schedule PDF (Gemini + pdfplumber fallback)

# Ensure PDF bytes are available
if schedule_pdf_bytes is None:
    raise ValueError("Footing Schedule PDF not uploaded. Use the upload UI and click 'Validate & Proceed'.")

# --- Helper Functions ---
def parse_rebar_string(rebar_str):
    if pd.isna(rebar_str) or not isinstance(rebar_str, str) or rebar_str.strip() == '':
        return None, None
    match = re.match(r'\((\d+)\)\s*#(\d+)', rebar_str.strip())
    if match:
        return int(match.group(1)), int(match.group(2))
    return None, None

def convert_to_decimal_feet(dimension_str):
    if pd.isna(dimension_str) or not isinstance(dimension_str, str) or dimension_str.strip() == '':
        return None
    s = dimension_str.strip().replace('"', '')
    if "'" in s:
        parts = s.split("'")
        ft = float(parts[0]) if parts[0].strip() else 0.0
        inch = float(parts[1].strip()) if len(parts) > 1 and parts[1].strip() else 0.0
        return ft + inch / 12.0
    try:
        val = float(s)
        return val / 12.0 if val > 12 else val  # Heuristic: large = inches
    except:
        return None

# --- Gemini API Extraction ---
print("\nAttempting smart extraction with Gemini API...")

schedule_df = pd.DataFrame()  # Default empty
source = "None"

try:
    # Improved prompt for table + diagrams
    prompt_schedule = [
        "You are an expert structural engineer analyzing this Footing Schedule PDF.",
        "The PDF may contain tables AND/OR diagrams/images of footings with drawn dimensions, rebar callouts, arrows, and labels.",
        "Extract rebar details for EVERY footing mark (F01, F02, etc.) from tables or diagrams.",
        "For each rebar line:",
        "  - Mark: e.g. F01",
        "  - Qty: integer number of bars (from (6) #5 etc.)",
        "  - Bar Size: integer (e.g. 5 for #5)",
        "  - Length_ft: bar length in decimal feet — read from dimension arrows/text in diagram or table (convert 5'-0\" → 5.0, 60\" → 5.0)",
        "If 'EACH WAY', 'EW', 'UNO' or similar → duplicate the entry (one for footing length, one for width).",
        "Output **ONLY** a strict JSON array — no markdown, no extra text, no explanations:",
        "[{\"Mark\":\"F01\",\"Qty\":6,\"Bar Size\":5,\"Length_ft\":5.0}, ...]"
    ]

    # Send PDF as multimodal content
    response = genai.GenerativeModel('gemini-2.5-flash').generate_content(
        prompt_schedule + [{"mime_type": "application/pdf", "data": schedule_pdf_bytes}]
    )

    raw_text = response.text.strip()

    # Clean and parse JSON
    import json, re
    # Corrected regex for stripping markdown code block fences
    cleaned = re.sub(r'^```json\n|```$', '', raw_text, flags=re.DOTALL)
    match = re.search(r'\[.*\](?![^{]*\{)', cleaned, re.DOTALL)

    if match:
        try:
            data = json.loads(match.group(0))
            if isinstance(data, list) and all(isinstance(item, dict) and 'Mark' in item for item in data):
                schedule_df = pd.DataFrame(data)
                schedule_df['Qty'] = pd.to_numeric(schedule_df['Qty'], errors='coerce').fillna(0).astype(int)
                schedule_df['Bar Size'] = pd.to_numeric(schedule_df['Bar Size'], errors='coerce').fillna(0).astype(int)
                schedule_df['Length_ft'] = pd.to_numeric(schedule_df['Length_ft'], errors='coerce').fillna(0.0)
                source = "Gemini API"
                print("\033[92mGemini API extraction successful!\033[0m")
            else:
                raise ValueError("Invalid structure")
        except Exception as e:
            print(f"\033[93mGemini JSON parsing failed: {e}\033[0m")
            print(f"Raw Gemini response (first 300 chars): {raw_text[:300]}...")
    else:
        print("\033[93mNo valid JSON array found in Gemini response\033[0m")
        print(f"Raw response: {raw_text[:300]}...")

except Exception as e:
    print(f"\033[91mGemini API error: {e}\033[0m")

# --- pdfplumber Fallback ---
if schedule_df.empty:
    print("\nFalling back to pdfplumber parsing...")
    source = "pdfplumber fallback"
    try:
        with pdfplumber.open(io.BytesIO(schedule_pdf_bytes)) as pdf:
            all_data = []
            for page in pdf.pages:
                tables = page.extract_tables()
                for table in tables:
                    if table:
                        all_data.extend(table)

        raw_df = pd.DataFrame(all_data)
        footing_rows = raw_df[raw_df[0].astype(str).str.startswith('F', na=False)].copy()

        if footing_rows.empty:
            raise ValueError("No footing marks found in PDF.")

        # Assume columns: Mark, Length, Width, Thickness, Bottom, Top, ...
        if len(footing_rows.columns) >= 6:
            footing_rows.columns = ['Mark', 'Length_str', 'Width_str', 'Thickness', 'Bottom_str', 'Top_str'] + \
                                   [f"Extra{i}" for i in range(6, len(footing_rows.columns))]

            final_data = []
            for _, row in footing_rows.iterrows():
                mark = row['Mark']
                L_ft = convert_to_decimal_feet(row['Length_str'])
                W_ft = convert_to_decimal_feet(row['Width_str'])

                for layer, col in [('Bottom', 'Bottom_str'), ('Top', 'Top_str')]:
                    qty, size = parse_rebar_string(row.get(col))
                    if qty is not None and size is not None:
                        if L_ft is not None:
                            final_data.append({'Mark': mark, 'Qty': qty, 'Bar Size': size, 'Length_ft': L_ft})
                        if W_ft is not None:
                            final_data.append({'Mark': mark, 'Qty': qty, 'Bar Size': size, 'Length_ft': W_ft})

            schedule_df = pd.DataFrame(final_data)
            print("\033[92mpdfplumber fallback successful\033[0m")
        else:
            raise ValueError("Unexpected column count in pdfplumber output")

    except Exception as e:
        print(f"\033[91mpdfplumber failed: {e}\033[0m")
        schedule_df = pd.DataFrame(columns=['Mark', 'Qty', 'Bar Size', 'Length_ft'])

# --- Final Display ---
print(f"\nSource: \033[94m{source}\033[0m")
print(f"Extracted {len(schedule_df)} rebar entries (including Each Way duplication).")

if not schedule_df.empty:
    # Nice styled table
    styled = schedule_df.style \
        .set_properties(**{'text-align': 'center', 'border': '1px solid #ccc'}) \
        .set_table_styles([
            {'selector': 'th', 'props': [('font-weight', 'bold'), ('background-color', '#f0f0f0')]},
            {'selector': 'td', 'props': [('padding', '6px'), ('background-color', 'transparent')]}
        ]) \
        .format(precision=2)

    display(styled)
else:
    print("\033[91mNo rebar details extracted from Schedule PDF.\033[0m")

# Optional raw Gemini JSON (debug)
if 'raw_schedule_data_from_gemini' in globals() and raw_schedule_data_from_gemini:
    #@title Click to see raw Gemini JSON response (debug only)
    print(json.dumps(raw_schedule_data_from_gemini, indent=2))

## 5. Calculation – Merge, Ignore Missing, Compute Weight

In [ ]:
# @title Calculate Rebar Weight and Handle Missing Entries

# Ensure footing_counts_df and schedule_df are available
if 'footing_counts_df' not in globals() or footing_counts_df.empty:
    raise ValueError("footing_counts_df not found or is empty. Please run Q1 first.")
if 'schedule_df' not in globals() or schedule_df.empty:
    warnings.warn("\033[93mWarning: schedule_df not found or is empty. Proceeding with available footing counts. Rebar weight will be 0 for all entries.\033[0m")

# 1. Merge the footing_counts_df (from Plan PDF) with the schedule_df (from Schedule PDF).
# Use a left join to keep all footing marks from the Plan PDF.
final_rebar_df = pd.merge(footing_counts_df, schedule_df, on='Mark', how='left')

# 2. Add 'Multiplier' column (default to 4)
final_rebar_df['Multiplier'] = DEFAULT_MULTIPLIER

# 3. Create a 'Notes' column for error handling and information
final_rebar_df['Notes'] = ''

# Identify marks from Plan PDF that have no corresponding rebar details in Schedule PDF
# These are rows where 'Qty' is NaN after the left merge, indicating no match from schedule_df
marks_missing_schedule_entry = final_rebar_df[final_rebar_df['Qty'].isna()]['Mark'].unique()

if len(marks_missing_schedule_entry) > 0:
    # Using ANSI escape codes for yellow warning text
    warnings.warn(
        f"\033[93mWarning: Footing marks from Plan PDF missing in Schedule: {', '.join(marks_missing_schedule_entry)}. "
        "Their rebar weight contribution will be assumed as zero. Verify Schedule PDF for completeness.\033[0m"
    )
    # Update 'Notes' column for these missing entries
    final_rebar_df.loc[final_rebar_df['Mark'].isin(marks_missing_schedule_entry), 'Notes'] = \
        "Ignored: No schedule entry found in Footing Schedule PDF."

# Fill NaN values in numerical columns that come from schedule_df with 0
# This ensures that footings with no schedule entry contribute 0 to the total weight
final_rebar_df['Qty'] = final_rebar_df['Qty'].fillna(0).astype(int)
final_rebar_df['Bar Size'] = final_rebar_df['Bar Size'].fillna(0).astype(int)
final_rebar_df['Length_ft'] = final_rebar_df['Length_ft'].fillna(0.0)

# 4. Map 'Bar Size' to 'Unit Weight (lbs/ft)'
# Use .map() and fill any unmapped bar sizes (e.g., bar size 0 or undefined) with 0.
final_rebar_df['Unit Weight (lbs/ft)'] = final_rebar_df['Bar Size'].map(rebar_unit_weights_lbs_per_foot).fillna(0.0)

# Check for unknown bar sizes (those mapped to 0 because they weren't in the unit_weights dict)
unknown_bar_sizes = final_rebar_df[(final_rebar_df['Bar Size'] != 0) & (final_rebar_df['Unit Weight (lbs/ft)'] == 0)]['Bar Size'].unique()
if len(unknown_bar_sizes) > 0:
    warnings.warn(f"\033[93mWarning: Unknown bar sizes encountered: {', '.join(map(str, unknown_bar_sizes))}. Their rebar weight contribution will be assumed as zero.\033[0m")
    # Add a note for unknown bar sizes
    final_rebar_df.loc[final_rebar_df['Bar Size'].isin(unknown_bar_sizes), 'Notes'] = \
        final_rebar_df.loc[final_rebar_df['Bar Size'].isin(unknown_bar_sizes), 'Notes'] + \
        ("; Unknown bar size, weight set to 0." if final_rebar_df['Notes'].ne('') else "Unknown bar size, weight set to 0.")


# 5. Apply the formula to calculate 'Rebar Weight (lb)'
# Formula: Count × Multiplier × Qty × Length(ft) × Unit Weight (lbs/ft)
final_rebar_df['Rebar Weight (lb)'] = (
    final_rebar_df['Count'] *
    final_rebar_df['Multiplier'] *
    final_rebar_df['Qty'] *
    final_rebar_df['Length_ft'] * # Length is already in feet
    final_rebar_df['Unit Weight (lbs/ft)']
)

# Ensure calculated weight is non-negative
final_rebar_df['Rebar Weight (lb)'] = final_rebar_df['Rebar Weight (lb)'].clip(lower=0)

print("Rebar weight calculation complete and missing entries handled.")

In [ ]:
DEFAULT_MULTIPLIER = 4

# Standard rebar unit weights in lbs/ft
rebar_unit_weights_lbs_per_foot = {
    3: 0.376,   # #3 bar
    4: 0.668,   # #4 bar
    5: 1.043,   # #5 bar
    6: 1.502,   # #6 bar
    7: 2.044,   # #7 bar
    8: 2.670,   # #8 bar
    9: 3.400,   # #9 bar
    10: 4.303,  # #10 bar
    11: 5.313,  # #11 bar
    14: 7.650,  # #14 bar
    18: 13.600  # #18 bar
}

print("Global constants DEFAULT_MULTIPLIER and rebar_unit_weights_lbs_per_foot defined.")

In [ ]:
# @title Display Results and Export CSVs

# 1. Display the detailed resulting table (select relevant columns and style for presentation)
print("\nDetailed Rebar Weight Calculation Table (First 10 rows):")
display(final_rebar_df[[
    'Mark', 'Count', 'Multiplier', 'Qty', 'Bar Size', 'Length_ft',
    'Unit Weight (lbs/ft)', 'Rebar Weight (lb)', 'Notes'
]].head(10).style.set_properties(**{'text-align': 'left'}).set_table_styles([dict(selector='th', props=[('text-align', 'left')])]))

# 2. Calculate and present the grand total rebar weight
grand_total_rebar_weight_lbs = final_rebar_df['Rebar Weight (lb)'].sum()
print(f"\n\033[1mGrand Total Rebar Weight: {grand_total_rebar_weight_lbs:.2f} lbs\033[0m") # Bold total

# 3. Export the final results to rebar_weight_results.csv
output_filename = 'rebar_weight_results.csv'
final_rebar_df.to_csv(output_filename, index=False)
print(f"\n\033[92mDetailed rebar weight results exported to '{output_filename}'.\033[0m") # Green success

# 4. Provide instructions for downloading output files
print("\n\033[1mNotebook ready! Download CSVs from the Files panel (left sidebar) or by running the following commands:\033[0m")
print(f"files.download('footing_counts.csv')")
print(f"files.download('{output_filename}')")